# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and accessible at:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

---

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load Croissant metadata and instantiate the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata and view summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published on: {metadata.datePublished}")

---
## 2. Data Overview

### Available Record Sets

List all record sets defined in the Croissant schema along with their `@id`s, and explore their fields and columns. All references are by their Croissant `@id` for reproducible extraction and processing.

In [ ]:
# Examine all record sets, fields, and columns (all referenced by @id)
print("\nAvailable Record Sets:\n" + "-" * 28)
record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    # List fields for each record set
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id: {f.id}, dataType: {f.data_type})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (@id: {col.id}, dataType: {col.data_type})")
    print()

print(f"All Record Set @IDs: {record_set_ids}")

### Sample Records by RecordSet `@id`

Use the record set `@id` from above to print a sample record. This illustrates how all Croissant entities are referenced by their `@id` throughout the notebook.

In [ ]:
# Print one sample record from each RecordSet by @id
for recset_id in record_set_ids:
    print(f"\nFirst record in RecordSet @id='{recset_id}':")
    try:
        recs = dataset.records(record_set=recset_id)
        for idx, rec in enumerate(recs):
            print(rec)
            if idx >= 0:  # Only one record
                break
    except Exception as e:
        print(f"  [Error accessing records: {e}]")

---
## 3. Data Extraction

Extract all records from each record set by their `@id` for programmatic analysis, and assemble them into pandas DataFrames. Use the columns/fields' `@id` as DataFrame column names where applicable.

In [ ]:
# Extract all data from each record set by @id
dfs = {}
for rs_id in record_set_ids:
    print(f"\nLoading records for RecordSet '{rs_id}'...")
    recs = list(dataset.records(record_set=rs_id))
    if len(recs):
        dfs[rs_id] = pd.DataFrame(recs)
        print(f"  -> {len(dfs[rs_id])} records loaded, columns: {dfs[rs_id].columns.tolist()}")
    else:
        print(f"  [No records found for {rs_id}]")

# Preview one of the main dataframes
if dfs:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns for main data DataFrame (@id='{main_rs_id}'):")
    print(dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())

---
## 4. Exploratory Data Analysis (EDA)

Select numeric and grouping fields by their Croissant `@id` for demonstration (e.g., GAD-7, PHQ-9, Age, Village). Filter, normalize, and aggregate records using only the proper `@id` column names.

In [ ]:
# We'll use the first RecordSet for analysis
record_set_to_analyze = record_set_ids[0]
df = dfs[record_set_to_analyze]

# List all float/integer fields using Croissant schema (@id and type)
numeric_ids = []
grouping_ids = []
for rs in record_sets:
    if rs.id == record_set_to_analyze:
        if hasattr(rs, 'fields'):
            for f in rs.fields:
                if f.data_type in ['Integer', 'Float', 'Number']:
                    numeric_ids.append(f.id)
                elif f.data_type in ['Text']:
                    grouping_ids.append(f.id)
        if hasattr(rs, 'columns') and rs.columns:
            for c in rs.columns:
                if c.data_type in ['Integer', 'Float', 'Number']:
                    numeric_ids.append(c.id)
                elif c.data_type in ['Text']:
                    grouping_ids.append(c.id)

print(f"Numeric field (@ids): {numeric_ids}")
print(f"Text (grouping) field (@ids): {grouping_ids}")

# --- Choose a representative numeric and grouping field by @id ---
if len(numeric_ids) == 0:
    raise ValueError("No numeric fields found for EDA.")

numeric_field_id = numeric_ids[0]   # e.g. '@gad7_score' or similar
group_field_id = grouping_ids[0] if grouping_ids else None

print(f"Using numeric_field_id: {numeric_field_id}")
if group_field_id:
    print(f"Using group_field_id: {group_field_id}")

# Remove outliers: filter values > lower bound
threshold = df[numeric_field_id].mean() - 2*df[numeric_field_id].std()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (remove outliers below)")
display(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    grouped_df.columns = [group_field_id, f"mean_{numeric_field_id}"]
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

---
## 5. Visualization

Visualize the filtered, normalized numeric data and group means using the discovered Croissant field `@id` as the column key.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Normalized values
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
plt.title(f"Distribution of Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.show()

# If group field exists, plot group means
if group_field_id and group_field_id in filtered_df.columns:
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=f"mean_{numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

---
## 6. Conclusion

- The dataset, referenced fully by Croissant schema and processed only via `@id` for all record sets and fields, provides a reproducible framework for analysis.
- We loaded, explored, and visualized survey data, focusing on key mental health numeric indicators (e.g., GAD-7, PHQ-9, PSQ) and demographic partitions, all with field provenance per Croissant standards.
- This notebook can be extended for domain-specific analyses and machine learning applications, building upon the FAIR² and mlcroissant ecosystem.